# 🌿 Plant Disease Detection — FIXED Training Notebook

**No Google Drive needed! Dataset downloads directly from Kaggle inside Colab.**

**Key fixes from previous version:**
1. ✅ Downloads PlantVillage dataset from Kaggle — no Drive upload needed  
2. ✅ Uses `mobilenet_v2.preprocess_input()` — correct normalization for MobileNetV2  
3. ✅ Class weights to prevent all predictions collapsing to one class  
4. ✅ Saves as `best_model.keras` (matches app config)  
5. ✅ Verification step confirms all classes predict before download  

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. You need a **Kaggle account** + API key (free). Instructions in Step 2.

## Step 1 — Verify GPU

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Setup Kaggle & Download PlantVillage Dataset

**How to get your Kaggle API key (one-time setup):**
1. Go to [kaggle.com](https://www.kaggle.com) → Sign in → Click your profile picture → **Settings**
2. Scroll to **API** section → Click **Create New Token**
3. A file called `kaggle.json` will download to your computer
4. Run the cell below — it will ask you to **upload** that `kaggle.json` file

In [ ]:
import os

# ── Upload your kaggle.json API key ───────────────────────────────────────
from google.colab import files
print('📂 Please upload your kaggle.json file when prompted...')
uploaded = files.upload()   # opens file picker — upload kaggle.json here

# Move to the correct location
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle API key configured')

In [ ]:
# ── Download PlantVillage dataset from Kaggle (≈1.1 GB, takes ~2-3 min) ──
print('Downloading PlantVillage dataset from Kaggle...')
!pip install -q kaggle
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/plantvillage --unzip

# Find the color subfolder automatically
import glob

# Look for the training folder (contains class subfolders)
# The Kaggle dataset structure: New Plant Diseases Dataset(Augmented)/train/
possible_roots = [
    '/content/plantvillage/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train',
    '/content/plantvillage/New Plant Diseases Dataset(Augmented)/train',
    '/content/plantvillage/train',
    '/content/plantvillage',
]

DATASET_DIR = None
for path in possible_roots:
    if os.path.isdir(path):
        subdirs = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
        if len(subdirs) >= 5:   # at least 5 class folders
            DATASET_DIR = path
            break

if DATASET_DIR is None:
    # Fallback: walk to find the folder with the most subdirs
    best, best_count = None, 0
    for root, dirs, files in os.walk('/content/plantvillage'):
        if len(dirs) > best_count:
            best_count = len(dirs)
            best = root
    DATASET_DIR = best

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f'\n✅ Dataset ready at: {DATASET_DIR}')
print(f'   {len(classes)} classes: {classes}')

## Step 3 — Configuration

In [ ]:
MODEL_SAVE_PATH  = '/content/best_model.keras'
TARGET_SIZE      = (224, 224)
BATCH_SIZE       = 32
EPOCHS_PHASE1    = 15
EPOCHS_PHASE2    = 15
VALIDATION_SPLIT = 0.2

print('Config set ✅')
print(f'  Dataset      : {DATASET_DIR}')
print(f'  Image size   : {TARGET_SIZE}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Phase 1 epochs: {EPOCHS_PHASE1}')
print(f'  Phase 2 epochs: {EPOCHS_PHASE2}')

## Step 4 — Data Generators

> **Critical fix:** Uses `preprocess_input()` (maps to [-1,1]) instead of `rescale=1/255` ([0,1]).  
> Using the wrong normalization causes all predictions to collapse to one class.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    shear_range=0.1,
    fill_mode='nearest',
)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=VALIDATION_SPLIT,
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR, target_size=TARGET_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True, seed=42,
)
val_gen = val_datagen.flow_from_directory(
    DATASET_DIR, target_size=TARGET_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False, seed=42,
)

class_names = list(train_gen.class_indices.keys())
num_classes  = len(class_names)
print(f'✅ Classes: {num_classes}  |  Train: {train_gen.samples}  |  Val: {val_gen.samples}')

## Step 5 — Compute Class Weights

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels   = train_gen.classes
unique_classes = np.unique(train_labels)
weights = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
class_weight_dict = dict(zip(unique_classes, weights))

print('✅ Class weights computed:')
for idx, w in class_weight_dict.items():
    print(f'  [{idx:2d}] {class_names[idx]}: {w:.3f}')

## Step 6 — Save Class Names JSON

In [ ]:
import json

index_to_class = {str(v): k for k, v in train_gen.class_indices.items()}
with open('/content/class_names.json', 'w') as f:
    json.dump(index_to_class, f, indent=2)
print('✅ class_names.json saved')
print(json.dumps(index_to_class, indent=2))

## Step 7 — Build Model

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, Input

base_model = MobileNetV2(input_shape=(*TARGET_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs  = Input(shape=(*TARGET_SIZE, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(512, activation='relu')(x)
x       = layers.Dropout(0.4)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model   = models.Model(inputs, outputs, name='plant_disease_mobilenetv2')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
print(f'Model built ✅  |  Params: {model.count_params():,}')

## Step 8 — Phase 1: Feature Extraction (Frozen Base)

In [ ]:
callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint('/content/best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',
        patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
        factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print('=== PHASE 1: Feature Extraction ===')
history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_PHASE1, callbacks=callbacks_p1,
    class_weight=class_weight_dict, verbose=1,
)
print(f'\nBest Phase 1 val_accuracy: {max(history1.history["val_accuracy"])*100:.2f}%')

## Step 9 — Phase 2: Fine-Tuning (Top 50 Layers)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint('/content/best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',
        patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
        factor=0.3, patience=3, min_lr=1e-8, verbose=1),
]

print('=== PHASE 2: Fine-Tuning ===')
history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_PHASE2, callbacks=callbacks_p2,
    class_weight=class_weight_dict, verbose=1,
)
print(f'\nBest Phase 2 val_accuracy: {max(history2.history["val_accuracy"])*100:.2f}%')

## Step 10 — Verify All Classes Predict Correctly

In [ ]:
from PIL import Image

best_model = tf.keras.models.load_model('/content/best_model.keras')
print('=== VERIFICATION: 1 sample per class ===')
correct = 0
total   = 0

for class_name in sorted(os.listdir(DATASET_DIR)):
    class_dir = os.path.join(DATASET_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    imgs = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if not imgs:
        continue
    arr  = np.array(Image.open(os.path.join(class_dir, imgs[0])).convert('RGB').resize(TARGET_SIZE), np.float32)
    probs = best_model.predict(np.expand_dims(preprocess_input(arr), 0), verbose=0)[0]
    pred_idx  = int(np.argmax(probs))
    pred_name = index_to_class[str(pred_idx)]
    ok = (pred_name == class_name)
    correct += int(ok)
    total   += 1
    print(f'{"✅" if ok else "❌"} {class_name[:38]:38s} → {pred_name[:30]} ({probs[pred_idx]*100:.1f}%)')

print(f'\nSample accuracy: {correct}/{total} = {correct/total*100:.1f}%')
print('✅ Ready to download!' if correct/total >= 0.5 else '⚠️  Low — consider more epochs.')

## Step 11 — Download Model Files

In [ ]:
from google.colab import files

print('Downloading best_model.keras ...')
files.download('/content/best_model.keras')
print('Downloading class_names.json ...')
files.download('/content/class_names.json')
print('\n✅ Done!')
print('Copy both files into:  plant_disease_prediction/models/saved_model/')